<a href="https://colab.research.google.com/github/aarya2400/AI-career/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q diffusers transformers accelerate torch torchvision opencv-python pillow gradio timm

In [6]:
import torch
import numpy as np
from PIL import Image
import gradio as gr
import cv2

from diffusers import StableDiffusionInpaintPipeline
from transformers import CLIPProcessor, CLIPModel

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [7]:
# Inpainting model
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to("cuda")

pipe.enable_attention_slicing()

# CLIP model for cloth understanding
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [8]:
# def generate_mask(image):
#     img = np.array(image)

#     # Convert to grayscale
#     gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

#     # Threshold (basic segmentation)
#     _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

#     # Morphology to smooth
#     kernel = np.ones((5,5), np.uint8)
#     mask = cv2.dilate(thresh, kernel, iterations=2)

#     mask = Image.fromarray(mask)

#     return mask

def generate_mask(image):
    image = image.resize((512, 512))

    mask = np.zeros((512, 512), dtype=np.uint8)

    # Upper body region (tuned)
    mask[120:420, 140:370] = 255

    return Image.fromarray(mask)

In [9]:
# labels = [
#     "red shirt", "blue jeans", "black jacket",
#     "white t-shirt", "formal suit", "hoodie",
#     "dress", "kurta", "saree"
# ]

# def get_cloth_description(cloth_img):
#     inputs = clip_processor(
#         text=labels,
#         images=cloth_img,
#         return_tensors="pt",
#         padding=True
#     ).to("cuda")

#     outputs = clip_model(**inputs)
#     logits_per_image = outputs.logits_per_image
#     probs = logits_per_image.softmax(dim=1)

#     idx = probs.argmax().item()
#     return labels[idx]

labels = [
    "black shirt", "white shirt", "blue jeans",
    "black jacket", "hoodie", "formal suit",
    "t-shirt", "kurta", "dress"
]

def get_cloth_description(cloth_img):
    cloth_img = cloth_img.resize((224, 224))

    inputs = clip_processor(
        text=labels,
        images=cloth_img,
        return_tensors="pt",
        padding=True
    ).to("cuda")

    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

    idx = probs.argmax().item()
    return labels[idx]

In [10]:
# def virtual_tryon_v2(person_img, cloth_img):
#     if person_img is None or cloth_img is None:
#         return None

#     # Step 1: Generate mask
#     mask = generate_mask(person_img)

#     # Step 2: Get cloth description
#     cloth_desc = get_cloth_description(cloth_img)

#     # Step 3: Prompt
#     prompt = f"A realistic photo of a person wearing {cloth_desc}, perfect fit, highly detailed"

#     negative_prompt = "blurry, distorted, bad quality, unrealistic body"

#     # Step 4: Generate output
#     result = pipe(
#         prompt=prompt,
#         negative_prompt=negative_prompt,
#         image=person_img,
#         mask_image=mask,
#         guidance_scale=7.5,
#         num_inference_steps=30
#     ).images[0]

#     return result

def virtual_tryon_v2(person_img, cloth_img):
    if person_img is None or cloth_img is None:
        return None

    # Resize inputs (VERY IMPORTANT)
    person_img = person_img.resize((512, 512))
    cloth_img = cloth_img.resize((512, 512))

    # Generate mask
    mask = generate_mask(person_img)

    # Get cloth description
    cloth_desc = get_cloth_description(cloth_img)

    # STRONG PROMPT (fixed)
    prompt = f"""
    A full body photo of a man wearing a {cloth_desc},
    perfect fit, realistic fabric, same person, same pose,
    highly detailed, natural lighting
    """

    negative_prompt = """
    cropped, half body, missing body parts, white shirt,
    blurry, distorted, unrealistic, bad anatomy
    """

    # Generate result
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=person_img,
        mask_image=mask,
        guidance_scale=9,
        num_inference_steps=40
    ).images[0]

    return result

In [11]:
demo = gr.Interface(
    fn=virtual_tryon_v2,
    inputs=[
        gr.Image(type="pil", label="Upload Person Image"),
        gr.Image(type="pil", label="Upload Cloth Image")
    ],
    outputs=gr.Image(label="Try-On Result"),
    title="👕 AI Virtual Try-On (Auto Mode)",
    description="Upload person + cloth image → AI does the rest"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a00b736327b1308db1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
